In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics")
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/Scfv_Toolkit")

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# Errors out during job runs and switched to using different compute
#import os
#path_boltz = os.path.abspath('/Workspace/Users/karthik.raj@bio-techne.com/boltz')
#os.chdir(path_boltz)

#%pip install --no-dependencies .

In [0]:
import sys
import os
import shutil
import tarfile
from pathlib import Path

# --- CONFIGURATION ---
# Permanent Storage (Unity Catalog Volume)
VOLUME_DIR = Path("/Volumes/sandbox/model_weights/protein_hunter")
# Fast Local Storage (Ephemeral SSD)
LOCAL_DIR = Path.home() / ".boltz"

# Create directories
os.makedirs(VOLUME_DIR, exist_ok=True)
if os.path.exists(LOCAL_DIR):
    # Clean slate for local dir to avoid half-extracted messes
    if os.path.islink(LOCAL_DIR):
        os.remove(LOCAL_DIR)
    else:
        shutil.rmtree(LOCAL_DIR)
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"📍 Hybrid Setup Initiated")
print(f"   Storage: {VOLUME_DIR}")
print(f"   Execution: {LOCAL_DIR}")

# --- HELPER FUNCTIONS ---
def install_from_volume():
    print("🔄 Found files in Volume! Linking to local SSD...")
    
    # 1. Link the Weights (Instant)
    # We don't copy the 3GB file, we just symlink it.
    vol_ckpt = VOLUME_DIR / "boltz2_conf.ckpt"
    local_ckpt = LOCAL_DIR / "boltz2_conf.ckpt"
    if vol_ckpt.exists():
        if os.path.exists(local_ckpt): os.remove(local_ckpt)
        os.symlink(vol_ckpt, local_ckpt)
        print("   ✅ Weights linked.")
    
    # 2. Extract the Molecules (Fast on SSD)
    # We copy the tarball and extract it locally
    vol_tar = VOLUME_DIR / "mols.tar"
    local_tar = LOCAL_DIR / "mols.tar"
    
    if vol_tar.exists():
        print("   ⏳ Copying CCD tarball to local SSD...")
        shutil.copy(vol_tar, local_tar)
        
        print("   📦 Extracting molecules locally (this will be fast)...")
        with tarfile.open(local_tar) as tar:
            tar.extractall(path=LOCAL_DIR)
        print("   ✅ CCD Data ready.")

def download_and_backup():
    print("⬇️ Files missing in Volume. Downloading fresh...")
    
    # Setup path to import the downloader
    repo_path = os.getcwd()
    sys.path.insert(0, os.path.join(repo_path, "boltz_ph"))
    try:
        from boltz.main import download_boltz2
    except ImportError:
        sys.path.insert(0, os.path.join(repo_path, "boltz_ph", "src"))
        from boltz.main import download_boltz2

    # Download to LOCAL SSD first (Fastest)
    download_boltz2(LOCAL_DIR)
    
    print("💾 Backing up to Volume for future runs...")
    # Copy the big files to the volume
    local_ckpt = LOCAL_DIR / "boltz2_conf.ckpt"
    local_tar = LOCAL_DIR / "mols.tar" # The downloader usually keeps the tar?
    
    # Note: boltz downloader extracts immediately. We might need to re-tar or just check what's there.
    # Usually it leaves boltz2_conf.ckpt and mols/ folder.
    
    # Backup Weights
    if local_ckpt.exists():
        shutil.copy(local_ckpt, VOLUME_DIR / "boltz2_conf.ckpt")
        print("   ✅ Weights backed up.")
        
    # Backup Molecules (Re-tar them for efficient storage)
    if (LOCAL_DIR / "mols").exists():
        print("   zzZ Compressing molecules for backup...")
        with tarfile.open(VOLUME_DIR / "mols.tar", "w") as tar:
            tar.add(LOCAL_DIR / "mols", arcname="mols")
        print("   ✅ CCD Data backed up.")

# --- EXECUTION LOGIC ---
# Check if the Volume already has the goods
if (VOLUME_DIR / "boltz2_conf.ckpt").exists() and (VOLUME_DIR / "mols.tar").exists():
    install_from_volume()
else:
    download_and_backup()

print("\n🚀 Ready to run! Environment is optimized for Databricks.")

In [0]:
import yaml
dbutils.widgets.text('path_task_yaml', "")
path_task_yaml = dbutils.widgets.get('path_task_yaml')
#path_task_yaml = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/inputs/gopher_alpha_snake.yaml" # ---------------- delete prior to full run
print("Path Task YAML: ", path_task_yaml)

if path_task_yaml != "":
    with open(path_task_yaml, 'r') as file:
        task_yaml = yaml.safe_load(file)

In [0]:
dbutils.widgets.text('run_boltz', '')
run_boltz = dbutils.widgets.get('run_boltz')
#run_boltz = 'apo' #--------------------------------------------------------delete prior to full run-------------------
print("Run Boltz ", run_boltz)
if run_boltz == "":
    raiseValueError(f"Please set run_boltz: {run_boltz} to either holo or apo")

In [0]:
dbutils.widgets.text('path_design_csv', '')
path_design_csv = dbutils.widgets.get('path_design_csv')
#path_design_csv = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/rfdiffusion3/designs/rfd3_test_7r2p1/af2_passed_boltz_input/af2_passed_designs.csv" #--------------------------- delete prior to full run --------------------------------
print("Path Design CSV: ", path_design_csv)

In [0]:
import pandas as pd
df_designs = pd.read_csv(path_design_csv, index_col = 0)
df_designs

In [0]:
from StrucTools import *
import biotite.structure as struc

def identify_binder_target_chains_from_structure(struc_input_path: str, seq_binder: str, seq_target: str):
    """ Helper function to assist in superposition and rmsd calculations by identifying the binder and target chain ids in the input structure. """
    atom_array_complex = extract_atom_array(struc_input_path, ca_only = False)
    atom_array_complex_protein = atom_array_complex[struc.filter_amino_acids(atom_array_complex)]
    seqs, seq_starts = struc.to_sequence(atom_array_complex_protein)
    chain_id_dict = {}
    for index, seq in enumerate(seqs):
        chain_id = chr(ord('A') + index)
        if str(seq) == seq_binder:
            chain_id_dict['binder'] = chain_id
        elif str(seq) == seq_target:
            chain_id_dict['target'] = chain_id
    return chain_id_dict 

def compute_rmsd_apo(input_path: str, boltz_design_path: str, chain_id_dict: dict):
    """ Extract Binder (APO) Coordinates from both input and boltz_design -> Align on APO Coordinates -> Compute RMSD on APO Coordinates"""

    atom_array_input = extract_atom_array(input_path, ca_only = False)
    atom_array_design = extract_atom_array(boltz_design_path, ca_only = False)
    
    # Identify respective binder chains in both inputs
    binder_chain_input = chain_id_dict['binder']
    binder_chain_design = "A"

    # Extract respective binder coordinates from both inputs
    binder_coords_input = atom_array_input[atom_array_input.chain_id == binder_chain_input].coord
    binder_coords_design = atom_array_design[atom_array_design.chain_id == binder_chain_design].coord
    aligned_design_binder_coords, transformation = struc.superimpose(binder_coords_input, binder_coords_design)
    rmsd = struc.rmsd(aligned_design_binder_coords, binder_coords_input)
    return rmsd

def compute_rmsd_holo(input_path: str, boltz_design_path: str, chain_id_dict: dict):
    """ Extract Target Coordinates from both input and boltz_design -> Align on Target Coordinates -> Compute RMSD on Binder Coordinates"""
    atom_array_input = extract_atom_array(input_path, ca_only = False)
    atom_array_design = extract_atom_array(boltz_design_path, ca_only = False)
    
    # Identify respective target chains in both inputs
    target_chain_input = chain_id_dict['target']
    target_chain_design = 'B'
    binder_chain_input = chain_id_dict['binder']
    binder_chain_design = "A"

    # Extract respective target atom array slices from both input and designed pdb
    atom_array_input_target = atom_array_input[atom_array_input.chain_id == target_chain_input]
    atom_array_design_target = atom_array_design[atom_array_design.chain_id == target_chain_design]
    
    # Generate transformation required to align design atom array's target to input atom array's target
    _, transformation = struc.superimpose(atom_array_input_target, atom_array_design_target)
    aligned_atom_array_design = transformation.apply(atom_array_design)

    # Extract Binder Coordinates from the input and the aligned design
    binder_coords_input = atom_array_input[atom_array_input.chain_id ==  binder_chain_input].coord
    binder_coords_design = aligned_atom_array_design[aligned_atom_array_design.chain_id == binder_chain_design].coord

    # Compute RMSD between the aligned design and input
    rmsd = struc.rmsd(binder_coords_design, binder_coords_input)
    return rmsd


def compute_binder_rmsd(input_path:str, boltz_design_path:str, run_boltz: str, chain_id_dict: dict):
    """ Compute RMSD between input and boltz_design on the binder chain with calculation varying based on  run_boltz value """
    if run_boltz == "apo":
        # Align on Binder, Measure on Binder
        rmsd = compute_rmsd_apo(input_path, boltz_design_path, chain_id_dict)
    elif run_boltz == "holo":
        # Align on Target, Measure on Binder
        rmsd = compute_rmsd_holo(input_path, boltz_design_path, chain_id_dict)
    else:
        raise ValueError("run_boltz must be either 'apo' or 'holo'")
    return rmsd

In [0]:
import os
from pathlib import Path

# Extract the path to design campaign folder which is always 2 levels above design csv path
level_save_dir_above_curr = 2
path_design_campaign = Path(path_design_csv).parents[level_save_dir_above_curr - 1]

# Create new folder to save boltz structures in
path_save_boltz_folder = path_design_campaign / f"boltz_{run_boltz}_structures/"
if not os.path.exists(path_save_boltz_folder):
    os.makedirs(path_save_boltz_folder)
    print(f"Created new folder to save Boltz structures: {path_save_boltz_folder}")
else:
    print(f"Boltz structures will be saved in existing folder: {path_save_boltz_folder}")

In [0]:
from StrucTools import *
from RunBoltz import *

all_design_metrics = []
for index, row in df_designs.iterrows():

    # Boltz Inputs: Independent of Holo or Apo
    design_name = row['design_name']
    seq_binder = row['seq_binder']
    seq_target = row['seq_target']
    num_models = 5

    # Apo Inputs
    if run_boltz == 'apo':
        seq_list = [seq_binder]
        msa_options = ['empty']
        ligand = ''
        # Contact Check Inputs: Epitope Residues (1-indexed) or empty list
        desired_epitope_residues = []
    
    # Holo Inputs
    elif run_boltz == 'holo':
        if path_task_yaml == "":
            raise ValueError("path_task_yaml must be provided for holo designs")

        seq_list = [seq_binder, seq_target]
        msa_options = ['empty', ""]
        ligand = task_yaml['ligand'] if "ligand" in task_yaml else ''
        print("Ligand: ", ligand)
        # Contact Check Inputs: Epitope Residues (1-indexed) or empty list
        desired_epitope_residues = task_yaml['hotspots'].split(',') if "hotspots" in task_yaml else []
        print("Epitope Residues: ", desired_epitope_residues)
    
    # Boltz Inputs: Dependent on the number of seqs passed for structure prediction (varies if apo or holo)
    template_paths = [''] * len(seq_list)
    entity_type = ['protein'] * len(seq_list)

    df_design_metrics = boltz_predict_analyze(design_name=design_name, volume_save_path=path_save_boltz_folder, seq_list=seq_list,msa_options=msa_options,template_paths=template_paths, entity_type=entity_type, desired_epitope_residues=desired_epitope_residues, num_models = num_models, ligand = ligand)

    ## Conduct RMSD Calculations (Varies between Apo or Holo): ---------------------------------------------------------------------------------------
    # Iterate through all the models predicted by Boltz and compute binder RMSD wrt to RFD Holo or Apo structure
    path_input_pdb = row['design_pdb_path']

    # Identify chain ID of binder and target in the input PDB
    chain_id_dict = identify_binder_target_chains_from_structure(struc_input_path= path_input_pdb, seq_binder = seq_binder, seq_target= seq_target)
    
    # Compute RMSD between input and boltz design (Calculation varies between holo or apo)
    df_design_metrics[f'rmsd_af2_boltz_{run_boltz}_binder'] = df_design_metrics['path_structure'].apply(
        lambda x: compute_binder_rmsd(input_path= path_input_pdb, boltz_design_path= x, run_boltz = run_boltz, chain_id_dict= chain_id_dict))
    
    df_design_metrics.rename(columns = {'path_structure' : f"path_boltz_{run_boltz}_structure"}, inplace= True)

    all_design_metrics.append(df_design_metrics)

df_all_design_metrics = pd.concat(all_design_metrics).reset_index(drop=True)
df_all_design_metrics


In [0]:
from pathlib import Path
if "chunk" in path_design_csv:
    chunk_id = path_design_csv.split("chunk_")[1].split('.')[0]
    design_csv_name = f"boltz_{run_boltz}_scored_designs_chunk_{chunk_id}.csv"
else:
    design_csv_name = f"boltz_{run_boltz}_scored_designs.csv"
path_parent_folder = Path(path_design_csv).parent
path_save_csv = str(path_parent_folder / design_csv_name)
# Save CSV of results to folder
df_all_design_metrics.to_csv(path_save_csv)